In [1]:
import pandas as pd
df = pd.read_csv("data.csv")

In [2]:
df

,id,index,match_id,period,timestamp,minute,second,event_type_id,event_type_name,possession,...,483440_interception_like_defensive_responsibility_probability,483427_interception_like_defensive_responsibility_probability,494440_interception_like_defensive_responsibility_probability,563697_interception_like_defensive_responsibility_probability,control_degree,is_transition,is_controlled_possession,player_possession_directness,possession_directness,location_bucket
0,b981cdbd-dec2-4760-a876-d9a270d55ec7,1,4068759,1,00:00,0,0,35,Starting XI,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,442bed77-90da-436e-ad7c-adebe51a2ef4,2,4068759,1,00:00,0,0,35,Starting XI,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4f95b807-be3e-4454-ae86-f41f2c07bd98,3,4068759,1,00:00:00.000,0,0,18,Half Start,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,98f74581-8d07-59c8-84e5-ec8b9e0a3512,4,4068759,1,00:00:00.000,0,0,18,Half Start,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,c58f5f95-bea5-4810-bf21-9411033e92b9,5,4068759,1,00:00:00.399,0,0,30,Pass,2,...,NaN,0.000035,NaN,NaN,1.0,False,True,-0.078626,0.0,midfield_third
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3887,028c9f3d-a3f1-4873-b065-7ed93c450a9f,2635,4068759,2,00:30:52.390,75,52,36,Tactical Shift,169,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3888,028c9f3d-a3f1-4873-b065-7ed93c450a9f,2635,4068759,2,00:30:52.390,75,52,36,Tactical Shift,169,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3889,028c9f3d-a3f1-4873-b065-7ed93c450a9f,2635,4068759,2,00:30:52.390,75,52,36,Tactical Shift,169,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3890,028c9f3d-a3f1-4873-b065-7ed93c450a9f,2635,4068759,2,00:30:52.390,75,52,36,Tactical Shift,169,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df["match_id"].nunique()

1

In [4]:
df.duplicated().sum()

np.int64(0)

In [8]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate event IDs:", df["id"].duplicated().sum())



Duplicate rows: 0
Duplicate event IDs: 660


In [5]:
# Jeden event może występować w kilku wierszach przez freeze_frame, więc usuwam duplikaty po id
events = df.drop_duplicates(subset="id").copy()

In [6]:
events[[
    "id",
    "period",
    "index",
    "team_name",
    "event_type_name"
]].isna().sum()

id                 0
period             0
index              0
team_name          0
event_type_name    0
dtype: int64

In [7]:
events[[
    "period",
    "index",
    "team_name",
    "event_type_name",
    "statsbomb_xg"
]].dtypes

period               int64
index                int64
team_name              str
event_type_name        str
statsbomb_xg       float64
dtype: object

In [9]:
team_PGM = "Pogoń Grodzisk Mazowiecki"
team_PB = "Polonia Bytom"

events = events.sort_values(["period", "index"])

score = {team_PGM: 0, team_PB: 0}
game_states = []

for _, row in events.iterrows():
    team = row["team_name"]
    opponent = team_PB if team == team_PGM else team_PGM

    diff = score[team] - score[opponent]
    game_states.append("Winning" if diff > 0 else "Losing" if diff < 0 else "Draw")

    if row["event_type_name"] == "Shot" and row["outcome_name"] == "Goal":
        score[team] += 1

events["game_state"] = game_states
shots = events[events["event_type_name"] == "Shot"].copy()

In [10]:
shots[[
        "minute",
        "second",
        "team_name",
        "player_name",
        "game_state",
        "statsbomb_xg",
        "outcome_name"
    ]]

,minute,second,team_name,player_name,game_state,statsbomb_xg,outcome_name
132,4,13,Polonia Bytom,Benedik Mioč,Draw,0.058596,Off T
164,5,6,Polonia Bytom,Kamil Wojtyra,Draw,0.100382,Blocked
524,13,38,Polonia Bytom,Kamil Wojtyra,Draw,0.028285,Off T
679,17,48,Polonia Bytom,Mikolaj Labojko,Draw,0.042174,Saved
782,21,13,Pogoń Grodzisk Mazowiecki,Kacper Los,Draw,0.197502,Goal
953,25,4,Pogoń Grodzisk Mazowiecki,Igor Korczakowski,Winning,0.054080,Blocked
958,25,7,Pogoń Grodzisk Mazowiecki,Damian Jaroń,Winning,0.063239,Blocked
1069,27,40,Polonia Bytom,Dominik Konieczny,Losing,0.043911,Saved
1073,27,43,Polonia Bytom,Lucjan Zielinski,Losing,0.035480,Goal
1103,29,34,Pogoń Grodzisk Mazowiecki,Igor Korczakowski,Draw,0.040728,Blocked


In [11]:
print(f"{team_PGM} {score[team_PGM]}:{score[team_PB]} {team_PB}")

Pogoń Grodzisk Mazowiecki 2:2 Polonia Bytom


In [12]:
events[
    (events["event_type_name"] == "Shot") &
    (events["outcome_name"] == "Goal")
][["minute", "team_name", "player_name"]]


,minute,team_name,player_name
782,21,Pogoń Grodzisk Mazowiecki,Kacper Los
1073,27,Polonia Bytom,Lucjan Zielinski
1409,36,Polonia Bytom,Maciej Wolski
1705,45,Pogoń Grodzisk Mazowiecki,Radoslaw Majewski


In [13]:
xg_state = shots.groupby(
    ["team_name", "game_state"]
)["statsbomb_xg"].sum()

xg_state

team_name                  game_state
Pogoń Grodzisk Mazowiecki  Draw          0.609015
                           Losing        0.016594
                           Winning       0.117319
Polonia Bytom              Draw          0.923124
                           Losing        0.079390
                           Winning       0.066237
Name: statsbomb_xg, dtype: float64

In [14]:
pgm_draw = (
    xg_state["Pogoń Grodzisk Mazowiecki", "Draw"]
    - xg_state["Polonia Bytom", "Draw"]
)

pgm_winning = (
    xg_state["Pogoń Grodzisk Mazowiecki", "Winning"]
    - xg_state["Polonia Bytom", "Losing"]
)

pgm_losing = (
    xg_state["Pogoń Grodzisk Mazowiecki", "Losing"]
    - xg_state["Polonia Bytom", "Winning"]
)

pb_draw = -pgm_draw
pb_winning = -pgm_losing
pb_losing = -pgm_winning

In [15]:
result = pd.DataFrame({
    "Drużyna": [
        team_PGM,
        team_PB
    ],
    "Remis": [
        pgm_draw,
        pb_draw
    ],
    "Wygrana": [
        pgm_winning,
        pb_winning
    ],
    "Przegrana": [
        pgm_losing,
        pb_losing
    ]
})

result = result.round(4)

result

,Drużyna,Remis,Wygrana,Przegrana
0,Pogoń Grodzisk Mazowiecki,-0.3141,0.0379,-0.0496
1,Polonia Bytom,0.3141,0.0496,-0.0379
